In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import multivariate_normal

In [ ]:
# create the phase labels df for training from the cleaned survey data
survey_df = pd.read_csv("final_clean_survey_risk.csv")



In [ ]:
df = survey_df.copy()
df['day'] = pd.to_datetime(df['day']).dt.normalize()

cycle_meta = []

for pid, g in df.groupby('name'):
    g = g.sort_values('day').reset_index(drop=True)

    # All menses start days = cycle boundaries
    m_starts = (
        g.loc[g['menzie_flag'] == 1, 'day']
        .sort_values()
        .tolist()
    )

    # Ovulation days (may be empty)
    o_days = (
        g.loc[g['ovulation_flag'] == 1, 'day']
        .sort_values()
        .tolist()
    )

    if len(m_starts) == 0:
        continue  # no cycles possible without menses

    for i, m_start in enumerate(m_starts):

        # ------------------------------------
        # DEFINE CYCLE END
        # ------------------------------------
        if i < len(m_starts) - 1:
            # Next menses defines the end of current cycle
            next_m_start = m_starts[i+1]
            cycle_end = next_m_start - pd.Timedelta(days=1)
            is_truncated = False
        else:
            # Last cycle → truncated (no future menses)
            cycle_end = g['day'].max()
            is_truncated = True

        # Window of this cycle
        cyc = g[(g['day'] >= m_start) & (g['day'] <= cycle_end)]

        # ------------------------------------
        # OVULATION ASSIGNMENT
        # ------------------------------------
        ovulations_in_cycle = (
            cyc.loc[cyc['ovulation_flag'] == 1, 'day']
            .sort_values()
            .tolist()
        )

        # A valid ovulation must occur strictly inside this cycle
        has_ovulation = (
            len(ovulations_in_cycle) == 1
        ) and (
            ovulations_in_cycle[0] > m_start
        ) and (
            ovulations_in_cycle[0] <= cycle_end
        )

        # ------------------------------------
        # COMPLETENESS CRITERION
        # ------------------------------------
        is_complete = (not is_truncated) and has_ovulation

        # Classification of incomplete types
        is_menses_only = (not is_complete) and (len(ovulations_in_cycle) == 0)
        is_ovulation_only = (not is_complete) and (len(ovulations_in_cycle) > 0)
        is_unlabeled = False  # you always have menses; no need for this flag

        cycle_meta.append({
            'name': pid,
            'cycle_id': i + 1,
            'cycle_start': m_start,
            'cycle_end': cycle_end,
            'is_truncated': is_truncated,
            'has_menses': True,
            'has_ovulation': has_ovulation,
            'is_complete': is_complete,
            'is_menses_only': is_menses_only,
            'is_ovulation_only': is_ovulation_only,
            'ovulation_day': ovulations_in_cycle[0] if has_ovulation else pd.NaT,
            'num_days': (cycle_end - m_start).days + 1
        })

cycle_table = pd.DataFrame(cycle_meta)
cycle_table


In [ ]:
print("\n================= CYCLE SUMMARY =================")

summary = {
    "Total cycles":            len(cycle_table),
    "Complete cycles":         cycle_table["is_complete"].sum(),
    "Menses-only cycles":      cycle_table["is_menses_only"].sum(),
    "Ovulation-only cycles":   cycle_table["is_ovulation_only"].sum(),
    "Truncated cycles":        cycle_table["is_truncated"].sum()
}

for k, v in summary.items():
    print(f"{k:25} {v}")

print("\n================= VALIDATION (column sums) =================")
print(cycle_table[[
    "is_complete",
    "is_menses_only",
    "is_ovulation_only",
    "is_truncated"
]].sum())


In [ ]:
# create the actual phase df

phase_rows_complete = []
phase_rows_partial = []

for pid, cyc in cycle_table.groupby('name'):
    # Subset participant survey rows
    g = survey_df[survey_df['name'] == pid].sort_values('day')

    for _, c in cyc.iterrows():
        cid = c['cycle_id']
        start = c['cycle_start']
        end = c['cycle_end']
        is_complete = c['is_complete']
        ovu = c['ovulation_day']
        
        # Restrict to this cycle window
        days = pd.date_range(start, end, freq='D')
        dfc = pd.DataFrame({'name': pid, 'day': days})
        dfc['cycle_id'] = cid
        
        # -------------------------------
        # COMPLETE CYCLES — FULL LABELING
        # -------------------------------
        if is_complete:
            dfc['phase'] = 'L'  # default luteal

            # Menses = 4 days
            m_end = start + pd.Timedelta(days=3)
            dfc.loc[dfc['day'].between(start, m_end), 'phase'] = 'M'

            # Ovulation day
            dfc.loc[dfc['day'] == ovu, 'phase'] = 'O'

            # Follicular = after menses until day before ovulation
            dfc.loc[
                (dfc['day'] > m_end) & (dfc['day'] < ovu),
                'phase'
            ] = 'F'

            # day_in_phase
            dfc['day_in_phase'] = dfc.groupby('phase').cumcount() + 1

            phase_rows_complete.append(dfc)

        # -------------------------------
        # PARTIAL CYCLES — ONLY M & O
        # -------------------------------
        else:
            dfp = dfc.copy()
            dfp['phase'] = np.nan

            # find actual menses days from survey_df
            m_days = pd.to_datetime(g.loc[g['menzie_flag'] == 1, 'day']).tolist()


            # label M (4 days)
            for m_start in m_days:
                if m_start >= start and m_start <= end:
                    m_end = m_start + pd.Timedelta(days=3)
                    dfp.loc[dfp['day'].between(m_start, m_end), 'phase'] = 'M'

            # label O days
            o_days = pd.to_datetime(g.loc[g['ovulation_flag'] == 1, 'day']).tolist()

            for o_day in o_days:
                if o_day >= start and o_day <= end:
                    dfp.loc[dfp['day'] == o_day, 'phase'] = 'O'

            dfp['day_in_phase'] = dfp.groupby('phase').cumcount() + 1

            phase_rows_partial.append(dfp)

# Combine
phase_daily_complete = pd.concat(phase_rows_complete).reset_index(drop=True)
phase_daily_partial  = pd.concat(phase_rows_partial).reset_index(drop=True)

phase_daily = pd.concat([phase_daily_complete, phase_daily_partial]).reset_index(drop=True)

phase_daily.head()


In [ ]:
# count how many days each phase lasts for each participant cycle
cycle_phase_durations = (
    phase_daily_complete
        .groupby(['name','cycle_id','phase'])
        .size()
        .reset_index(name='duration')
)

cycle_phase_durations.head()

In [ ]:
# Get distribution table
duration_dist = (
    cycle_phase_durations
        .groupby(['phase', 'duration'])
        .size()
        .unstack(fill_value=0)
)

duration_dist


In [ ]:
duration_probs = duration_dist.div(duration_dist.sum(axis=1), axis=0)
duration_probs


In [ ]:
duration_dict = {
    phase: duration_probs.loc[phase].to_dict()
    for phase in duration_probs.index
}

duration_dict


In [ ]:
# split cycles into complete cycles for training, complete for testing, partial for testing

# Make a copy to avoid modifying original
cycle_table_split = cycle_table.copy()

# Extract complete cycle rows
complete_cycles = cycle_table_split[cycle_table_split['is_complete']].copy()

# Fix random seed for reproducibility
np.random.seed(42)

# Randomly select 13 complete cycles for held-out testing
heldout_idx = np.random.choice(
    complete_cycles.index, 
    size=13, 
    replace=False
)

# Mark them
cycle_table_split['set'] = 'train'     # default
cycle_table_split.loc[heldout_idx, 'set'] = 'test_complete'

# All incomplete (non-complete) cycles are test_partial
cycle_table_split.loc[~cycle_table_split['is_complete'], 'set'] = 'test_partial'

# Extract lists
train_complete_cycles = cycle_table_split[
    (cycle_table_split['is_complete']) &
    (cycle_table_split['set'] == 'train')
]

test_complete_cycles = cycle_table_split[
    (cycle_table_split['is_complete']) &
    (cycle_table_split['set'] == 'test_complete')
]


test_partial_cycles = cycle_table_split[
    cycle_table_split['set'] == 'test_partial'
]

cycle_table_split.head()


In [ ]:
print("===== FINAL SPLIT =====")
print("Training complete cycles:", len(train_complete_cycles))
print("Held-out complete cycles:", len(test_complete_cycles))
print("Test partial cycles:", len(test_partial_cycles))
print("Total cycles:", len(cycle_table_split))


In [ ]:
# ---------------------------------------
# Build lookup sets (name, cycle_id) pairs
# ---------------------------------------

train_ids = {
    (row.name, row.cycle_id)
    for row in train_complete_cycles.itertuples(index=False)
}

test_complete_ids = {
    (row.name, row.cycle_id)
    for row in test_complete_cycles.itertuples(index=False)
}

test_partial_ids = {
    (row.name, row.cycle_id)
    for row in test_partial_cycles.itertuples(index=False)
}


# ---------------------------------------
# 1. TRAIN = complete cycles not held out
# ---------------------------------------

phase_train = phase_daily_complete[
    phase_daily_complete.apply(
        lambda r: (r['name'], r['cycle_id']) in train_ids,
        axis=1
    )
].reset_index(drop=True)


# ---------------------------------------
# 2. TEST (COMPLETE) = 13 held-out cycles
# ---------------------------------------

phase_test_complete = phase_daily_complete[
    phase_daily_complete.apply(
        lambda r: (r['name'], r['cycle_id']) in test_complete_ids,
        axis=1
    )
].reset_index(drop=True)


# ---------------------------------------
# 3. TEST (PARTIAL) = all incomplete cycles
# ---------------------------------------

phase_test_partial = phase_daily_partial[
    phase_daily_partial.apply(
        lambda r: (r['name'], r['cycle_id']) in test_partial_ids,
        axis=1
    )
].reset_index(drop=True)


# ---------------------------------------
# PRINT SUMMARY
# ---------------------------------------
print("===== PHASE DAILY SPLIT =====")
print("Train rows:          ", len(phase_train))
print("Test complete rows:  ", len(phase_test_complete))
print("Test partial rows:   ", len(phase_test_partial))

print("\nTrain cycles:        ", len(train_ids))
print("Test complete cycles:", len(test_complete_ids))
print("Test partial cycles: ", len(test_partial_ids))



In [ ]:
# Save to CSV
phase_train.to_csv("phase_daily_train.csv", index=False)
phase_test_complete.to_csv("phase_daily_test_complete.csv", index=False)
phase_test_partial.to_csv("phase_daily_test_partial.csv", index=False)

In [ ]:
# now create the combined biometric and survey risk data
survey_risk = pd.read_csv("final_clean_survey_risk.csv", parse_dates=["day"])
survey_risk['day'] = survey_risk['day'].dt.normalize()
biom = pd.read_csv("smoothed_biometrics.csv", parse_dates=["day"])
biom['day'] = biom['day'].dt.normalize()

In [ ]:
def merge_all(phase_df):
    out = (
        phase_df
        .merge(biom, on=['name','day'], how='left')
        .merge(
            survey_risk[['name','day','risk_obs','risk_latent_train','menzie_flag','ovulation_flag']],
            on=['name','day'], 
            how='left'
        )
    )
    return out


In [ ]:
train_df = merge_all(phase_train)
test_complete_df = merge_all(phase_test_complete)
test_partial_df = merge_all(phase_test_partial)
train_df.head(10)

In [ ]:
print("TRAIN shape:", train_df.shape)
print("TEST complete shape:", test_complete_df.shape)
print("TEST partial shape:", test_partial_df.shape)

print("Missing biometrics (train):")
print(train_df[['hrv_smooth','breath_smooth','temp_smooth']].isna().sum())

print("Missing risk or flags (train):")
print(train_df[['risk_obs','menzie_flag','ovulation_flag']].isna().sum())

# rows with missing risk_obs and menzie and ov flags were missing surveys

In [ ]:
# show the rows in train where risk_obs is NA
train_df[train_df['risk_obs'].isna()]

In [ ]:
# missing menzie and ovulation flags from train filled with zeros becasue these are missing survey coverage but still counted as complete cycles meaning that cycle had a menstruation and ovulation
event_cols = ['menzie_flag', 'ovulation_flag']

train_df[event_cols] = train_df[event_cols].fillna(0).astype(int)
test_complete_df[event_cols] = test_complete_df[event_cols].fillna(0).astype(int)





In [ ]:
#save to csv
train_df.to_csv("final_pomdp_train.csv", index=False)
test_complete_df.to_csv("final_pomdp_test_complete.csv", index=False)
test_partial_df.to_csv("final_pomdp_test_partial.csv", index=False)

In [ ]:
# read final dfs from csvs but drop col in all breath_smooth
train_df = pd.read_csv("final_pomdp_train.csv")
test_complete_df = pd.read_csv("final_pomdp_test_complete.csv")
test_partial_df = pd.read_csv("final_pomdp_test_partial.csv")
train_df = train_df.drop(columns=['breath_smooth'])
test_complete_df = test_complete_df.drop(columns=['breath_smooth'])
test_partial_df = test_partial_df.drop(columns=['breath_smooth'])

In [ ]:
# ------------------------------------------------------------
# 1. Standardize HRV + Temp per participant (critical)
# ------------------------------------------------------------
def compute_train_stats(train_df):
    stats = {}
    for pid, g in train_df.groupby("name"):
        mu_hrv  = g["hrv_smooth"].mean()
        sd_hrv  = g["hrv_smooth"].std()
        mu_temp = g["temp_smooth"].mean()
        sd_temp = g["temp_smooth"].std()

        # Handle degenerate cases
        if not np.isfinite(sd_hrv)  or sd_hrv  < 1e-8: sd_hrv  = 1.0
        if not np.isfinite(sd_temp) or sd_temp < 1e-8: sd_temp = 1.0

        stats[pid] = {
            "mu_hrv": mu_hrv, "sd_hrv": sd_hrv,
            "mu_temp": mu_temp, "sd_temp": sd_temp
        }
    return stats

def apply_standardization(df, train_stats):
    df = df.copy()
    for pid, g in df.groupby("name"):

        if pid in train_stats:
            ts = train_stats[pid]
            mu_hrv, sd_hrv   = ts["mu_hrv"], ts["sd_hrv"]
            mu_temp, sd_temp = ts["mu_temp"], ts["sd_temp"]
        else:
            # fallback for unseen participants
            mu_hrv  = df["hrv_smooth"].mean()
            sd_hrv  = df["hrv_smooth"].std()
            mu_temp = df["temp_smooth"].mean()
            sd_temp = df["temp_smooth"].std()

            if not np.isfinite(sd_hrv)  or sd_hrv  < 1e-8: sd_hrv  = 1.0
            if not np.isfinite(sd_temp) or sd_temp < 1e-8: sd_temp = 1.0

        df.loc[g.index, "hrv_smooth"]  = (g["hrv_smooth"]  - mu_hrv)  / sd_hrv
        df.loc[g.index, "temp_smooth"] = (g["temp_smooth"] - mu_temp) / sd_temp

    return df

def preprocess_test_df(df, train_stats):
    df = df.copy().sort_values(["name", "day"])

    df["true_onset"] = (df["menzie_flag"] == 1).astype(int)

    df = apply_standardization(df, train_stats)

    df["biometrics"] = df.apply(
        lambda row: np.array([row["hrv_smooth"], row["temp_smooth"]]),
        axis=1
    )

    return df

# apply standardization to the TRAIN SET ONLY
# Compute stats ONCE from training data
train_stats = compute_train_stats(train_df)

# Standardize training using training stats
train_std = apply_standardization(train_df, train_stats)

# ------------------------------------------------------------
# 2. Build HSMM Gaussian emissions from standardized TRAIN data
# ------------------------------------------------------------
em_train = train_std.dropna(subset=['hrv_smooth', 'temp_smooth']).copy()
phases = ['M', 'F', 'O', 'L']
emission_params = {}

for ph in phases:
    sub = em_train[em_train['phase'] == ph][['hrv_smooth', 'temp_smooth']]

    if len(sub) == 0:
        print(f"WARNING: no data for phase {ph}")
        continue

    mean_vec = sub.mean().values       # shape (2,)
    cov_mat  = sub.cov().values        # shape (2,2)

    # numerical safety
    cov_mat += np.eye(2) * 1e-6

    emission_params[ph] = {
        "mean": mean_vec,
        "cov": cov_mat
    }


print("\n=== HSMM Gaussian Emission Parameters (STANDARDIZED) ===")
for k,v in emission_params.items():
    print(f"\nPhase {k}:")
    print("Mean:", v['mean'])
    print("Covariance:\n", v['cov'])


In [ ]:
phase_order = ['M', 'F', 'O', 'L']
next_phase = {
    'M': 'F',
    'F': 'O',
    'O': 'L',
    'L': 'M'
}

hsmm_params = {
    "emissions": emission_params,
    "durations": duration_dict,
    "transitions": next_phase,
    "phases": phase_order
}

print("\nHSMM parameters ready.")

In [ ]:
# now build the risk transition and observation model

# -----------------------------------
# 1. Risk transition model P(R_{t+1} | R_t, Phase_t)
# -----------------------------------
train_df['day'] = pd.to_datetime(train_df['day'], errors='coerce')

# Use only rows where risk_latent_train is observed
rt = train_df[['name','day','phase','risk_latent_train']].dropna(subset=['risk_latent_train']).copy()
rt['risk_latent_train'] = rt['risk_latent_train'].astype(int)

# Sort by participant + day
rt = rt.sort_values(['name','day'])

# Build next-day risk and phase
rt['risk_next'] = rt.groupby('name')['risk_latent_train'].shift(-1)
rt['phase_next'] = rt.groupby('name')['phase'].shift(-1)
rt['day_next'] = rt.groupby('name')['day'].shift(-1)

# Keep only true day-to-day transitions
rt = rt[(rt['day_next'] == rt['day'] + pd.Timedelta(days=1))]
rt = rt.dropna(subset=['risk_next', 'phase_next'])
rt['risk_next'] = rt['risk_next'].astype(int)

phases = hsmm_params['phases']  # ['M','F','O','L']
risk_states = [0, 1]

# Laplace smoothing: start with 1 count in each cell
risk_trans_counts = {ph: np.ones((2, 2)) for ph in phases}
global_counts = np.ones((2, 2))

for _, row in rt.iterrows():
    ph = row['phase']
    r  = int(row['risk_latent_train'])
    r2 = int(row['risk_next'])
    if ph in risk_trans_counts:
        risk_trans_counts[ph][r, r2] += 1
    global_counts[r, r2] += 1

# Normalize rows to get probabilities
def row_normalize(mat):
    mat = mat.astype(float)
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return mat / row_sums

risk_transition = {ph: row_normalize(risk_trans_counts[ph]) for ph in phases}
risk_transition_global = row_normalize(global_counts)

print("Risk transition matrices per phase (rows = R_t, cols = R_{t+1}):")
for ph in phases:
    print(f"\nPhase {ph}:\n", risk_transition[ph])

print("\nGlobal risk transition matrix:\n", risk_transition_global)

In [ ]:

# ============================================================
# 1B. CARE-modified risk transition model
# ============================================================

boost = 0.25   # tune this

risk_transition_care = {}

for ph in phases:
    base = risk_transition[ph].copy()
    care = base.copy()

    # Only modify transitions out of high-risk (state 1)
    increase = min(base[1,0] + boost, 1.0)
    care[1,0] = increase
    care[1,1] = 1.0 - increase

    # low-risk (state 0) unchanged
    risk_transition_care[ph] = row_normalize(care)

print("\nCARE risk transition matrices:")
for ph in phases:
    print(f"\nPhase {ph}:\n", risk_transition_care[ph])


In [ ]:
# ============================================================
# ========== 3. NEW: Semi-Markov Phase Transitions ============
# ============================================================

duration_params = hsmm_params["durations"]     # Your HSMM durations
phase_index = {ph:i for i,ph in enumerate(phases)}
nP = len(phases)

phase_mean_duration = {}
phase_p_self = {}
phase_p_change = {}

for ph in phases:
    dist = duration_params[ph]   # {duration: prob}
    mean_dur = sum(d * p for d, p in dist.items())
    phase_mean_duration[ph] = mean_dur
    
    p_self = max((mean_dur - 1) / mean_dur, 0)
    p_change = 1 - p_self

    phase_p_self[ph] = p_self
    phase_p_change[ph] = p_change

# Build Phase Transition Matrix (4×4)
Phase_T = np.zeros((nP, nP))

# M → M/F
Phase_T[phase_index["M"], phase_index["M"]] = phase_p_self["M"]
Phase_T[phase_index["M"], phase_index["F"]] = phase_p_change["M"]

# F → F/O
Phase_T[phase_index["F"], phase_index["F"]] = phase_p_self["F"]
Phase_T[phase_index["F"], phase_index["O"]] = phase_p_change["F"]

# O → L only (duration=1 → p_self=0)
Phase_T[phase_index["O"], phase_index["O"]] = phase_p_self["O"]
Phase_T[phase_index["O"], phase_index["L"]] = phase_p_change["O"]

# L → L/M
Phase_T[phase_index["L"], phase_index["L"]] = phase_p_self["L"]
Phase_T[phase_index["L"], phase_index["M"]] = phase_p_change["L"]

print("Corrected Phase Transition Matrix:")
print(pd.DataFrame(Phase_T, index=phases, columns=phases))


In [ ]:
# ============================================================
# ========== 4. NEW: Build Correct Joint Transition ===========
# ============================================================

state_list = []
state_index = {}
idx = 0
for ph in phases:
    for r in risk_states:
        state_list.append((ph, r))
        state_index[(ph, r)] = idx
        idx += 1

def build_joint_matrix(Phase_T, risk_T_dict):
    """
    Phase_T:    4×4 matrix of P(ph'|ph)
    risk_T_dict: dict phase -> 2×2 risk transitions
    """
    nS = len(state_list)
    T = np.zeros((nS, nS))

    for ph in phases:
        ph_i = phase_index[ph]
        for r in risk_states:
            s = state_index[(ph, r)]

            for ph2 in phases:
                ph_j = phase_index[ph2]
                ph_prob = Phase_T[ph_i, ph_j]   # phase transition probability

                for r2 in risk_states:
                    r_prob = risk_T_dict[ph][r, r2]

                    s2 = state_index[(ph2, r2)]
                    T[s, s2] = ph_prob * r_prob

    return T

# Transitions per action
T_wait  = build_joint_matrix(Phase_T, risk_transition)
T_query = T_wait.copy()
T_alert = T_wait.copy()
T_care  = build_joint_matrix(Phase_T, risk_transition_care)

print("\nJoint Transition Matrix (WAIT):")
print(pd.DataFrame(T_wait))

In [ ]:
# -----------------------------------
# 2. Risk observation model P(risk_obs | Risk_latent)
# -----------------------------------

# Use only rows where both latent risk and observed risk are present
obs_df = train_df.dropna(subset=['risk_latent_train', 'risk_obs']).copy()
obs_df['risk_latent_train'] = obs_df['risk_latent_train'].astype(int)
obs_df['risk_obs'] = obs_df['risk_obs'].astype(int)

risk_obs_emission = {}

for r in risk_states:
    sub = obs_df[obs_df['risk_latent_train'] == r]
    if len(sub) == 0:
        p = 0.5  # fallback if no data
    else:
        p = sub['risk_obs'].mean()
    risk_obs_emission[r] = p

print("\nRisk observation model P(risk_obs=1 | R):")
for r, p in risk_obs_emission.items():
    print(f"R={r}: p={p:.3f}")


In [ ]:

# -----------------------------------
# 4. Observation model pieces
#    Biometrics: already Gaussian given Phase (hsmm_params['emissions'])
#    Symptoms: Bernoulli given Risk (risk_obs_emission)
# -----------------------------------

# Biometrics emission: X | (Phase=ph, Risk=r) ~ N(mean_ph, cov_ph)
# (independent of risk; we just reuse emission_params for each risk level)

biometric_emissions = emission_params # {phase: {'mean':..., 'cov':...}}

# Symptom emission: Y | (Phase, Risk=r) uses only R
# P(Y=1 | (ph, r)) = risk_obs_emission[r]
symptom_emission = {}
for ph in phases:
    for r in risk_states:
        symptom_emission[(ph, r)] = risk_obs_emission[r]

print("\nSymptom observation model P(Y=1 | Phase, Risk):")
for st, p in symptom_emission.items():
    print(f"{st}: p={p:.3f}")


In [ ]:
# ============================================================
# 5. Rewards
# ============================================================

# ALERT reward depends on timing error Δ computed externally
def alert_reward(delta):
    d = abs(delta)
    if d <= 1:
        return 10
    if 2 <= d <= 3:
        return 3
    if delta < -3:
        return -5
    if delta > 1:
        return -10
    return 0

reward_table = {
    "WAIT":  0,
    "QUERY": -1,
    "CARE":  -1,    # tunable
    "ALERT": alert_reward
}



In [ ]:
# -----------------------------------
# 5. Pack everything into a POMDP structure
# -----------------------------------

pomdp_model = {
    "states": state_list,
    "state_index": state_index,

    "T": {
        "WAIT":  T_wait,
        "QUERY": T_query,
        "CARE":  T_care,
        "ALERT": T_alert
    },

    "emissions": {
        "biometric": emission_params,
        "symptom": symptom_emission
    },

    "risk_transition_baseline": risk_transition,
    "risk_transition_care": risk_transition_care,

    "risk_obs_emission": risk_obs_emission,

    "reward": reward_table
}

print("\nPOMDP core model (with CARE action) is ready.")

In [ ]:
# ============================================================
# Preprocessing: convert raw test_df into per-day POMDP inputs
# ============================================================

def preprocess_test_df(df, train_stats):
    """
    Convert raw test dataframe for POMDP evaluation:
    - Sort by participant + day
    - Add true_onset flag
    - Standardize HRV using TRAIN μ,σ (per participant)
    - Build biometrics vector for emission model
    """
    df = df.copy().sort_values(["name", "day"])

    # ----- true onset label -----
    df["true_onset"] = (df["menzie_flag"] == 1).astype(int)

    # ----- Standardize using training stats -----
    df = apply_standardization(df, train_stats)

    # ----- Build biometrics vector -----
    df["biometrics"] = df["hrv_smooth"].apply(lambda x: np.array([x]))

    return df


In [ ]:


# ============================================================
# Belief update components
# ============================================================

def predict_belief(b, T):
    return b @ T

def obs_loglikelihood(obs, state, pomdp_model):
    ph, r = state

    # ----- biometrics log-likelihood -----
    mean = pomdp_model["emissions"]["biometric"][ph]["mean"]
    cov  = pomdp_model["emissions"]["biometric"][ph]["cov"]

    # CRITICAL: ensure obs["biometrics"] is a 1D numpy array (not wrapped in extra brackets)
    x = np.atleast_1d(obs["biometrics"]).flatten()
    mean = np.atleast_1d(mean).flatten()
    
    # Ensure cov is 2D
    cov = np.atleast_2d(cov)
    
    # Add small regularization to avoid singular matrix
    cov_regularized = cov + np.eye(cov.shape[0]) * 1e-8
    # # Reshape x to be 2D, explicitly (1 observation, 1 feature)
    # x_reshaped = x.reshape(1, -1)
    try:
        ll_biom = multivariate_normal.logpdf(
            x,
            mean=mean,
            cov=cov_regularized,
            allow_singular=True
        )
    except Exception as e:
        print(f"ERROR in logpdf: x shape {x.shape}, mean shape {mean.shape}, cov shape {cov.shape}")
        print(f"x={x}, mean={mean}, cov={cov}")
        ll_biom = -np.inf

    # ----- symptom log-likelihood -----
    y = obs["symptom"]

    if y is None:
        ll_sym = 0.0  # missing → uninformative
    else:
        p = pomdp_model["emissions"]["symptom"][(ph, r)]
        p = np.clip(p, 1e-6, 1 - 1e-6)  # avoid log(0)
        ll_sym = np.log(p) if y == 1 else np.log(1 - p)

    return ll_biom + ll_sym

def update_belief(b_pred, obs, pomdp_model):

    states = pomdp_model["states"]
    S = len(states)

    # ----- log prior -----
    log_b_pred = np.log(b_pred + 1e-20)

    # ----- log-likelihoods -----
    log_like = np.zeros(S)
    for i, s in enumerate(states):
        log_like[i] = obs_loglikelihood(obs, s, pomdp_model)

    # ----- unnormalized log posterior -----
    log_post = log_b_pred + log_like

    # ----- stable softmax normalization -----
    max_log = np.max(log_post)
    post = np.exp(log_post - max_log)
    post /= post.sum()

    return post


In [ ]:
# ============================================================
# Threshold policy
# ============================================================

def threshold_policy(b, pomdp_model, tau_risk, tau_onset, entropy_cut):
    # marginal risk prob
    p_risk = b[[i for i,(ph,r) in enumerate(pomdp_model["states"]) if r == 1]].sum()

    # marginal late luteal prob (onset soon)
    luteal_indices = [i for i, (ph, r) in enumerate(pomdp_model["states"]) if ph == "L"]
    p_late = b[luteal_indices].sum()

    if p_risk > tau_risk:
        return "CARE"
    if p_late > tau_onset:
        return "ALERT"

    # entropy as uncertainty measure
    entropy = -np.sum(b * np.log(b + 1e-12))
    if entropy > entropy_cut:
        return "QUERY"

    return "WAIT"


# ============================================================
# Run threshold policy for one cycle
# ============================================================

def run_cycle_threshold(cycle_df, pomdp_model, tau_risk, tau_onset, entropy_cut):

    S = len(pomdp_model["states"])
    # Initialize belief from the first day's phase
    first_phase = cycle_df["phase"].iloc[0]

    b = np.zeros(S)
    b[state_index[(first_phase, 0)]] = 0.5
    b[state_index[(first_phase, 1)]] = 0.5

    alerted_day = None
    rewards = 0
    query_count = 0

    # true onset day for cycle
    true_onset_days = cycle_df.loc[cycle_df["true_onset"] == 1, "day"]
    if len(true_onset_days) == 0:
        return None  # skip cycles with no onset label
    true_onset_day = pd.to_datetime(true_onset_days.iloc[0])

    for _, row in cycle_df.iterrows():

        # ----- choose action -----
        a = threshold_policy(b, pomdp_model, tau_risk, tau_onset, entropy_cut)

        if a == "QUERY":
            query_count += 1

        # ----- reward -----
        if a in ["WAIT", "QUERY", "CARE"]:
            rewards += pomdp_model["reward"][a]

        if a == "ALERT" and alerted_day is None:
            delta = (row["day"] - true_onset_day).days
            rewards += pomdp_model["reward"]["ALERT"](delta)
            alerted_day = row["day"]

        # ----- belief prediction -----
        T = pomdp_model["T"][a]
        b_pred = predict_belief(b, T)

        # ----- belief update using today's obs -----
        risk_obs_val = row["risk_obs"]
        if pd.isna(risk_obs_val):
            # missing symptom → uninformative likelihood
            symptom = None
        else:
            symptom = int(risk_obs_val)
        obs = {
            "biometrics": row["biometrics"],
            "symptom": symptom
        }
        b = update_belief(b_pred, obs, pomdp_model)

    # end-of-cycle outcome
    if alerted_day is None:
        delta = None
        rewards += -15  # penalty for no alert
    else:
        delta = (alerted_day - true_onset_day).days

    return {
        "cycle_id": cycle_df["cycle_id"].iloc[0],
        "alert_day": alerted_day,
        "delta": delta,
        "queries": query_count,
        "reward": rewards
    }


In [ ]:
# ============================================================
# Evaluate over a full dataset
# ============================================================

def evaluate_threshold(test_df_prep, pomdp_model, tau_risk, tau_onset, entropy_cut):

    results = []

    for cycle_id, cycle_df in test_df_prep.groupby("cycle_id"):
        out = run_cycle_threshold(cycle_df, pomdp_model, tau_risk, tau_onset, entropy_cut)
        if out is not None:
            results.append(out)

    df = pd.DataFrame(results)
    if len(df) == 0:
        return None

    # ----- Metrics -----
    alerted = df[df["delta"].notna()]
    timing_errors = alerted["delta"].abs()

    timely_alert_rate = (timing_errors <= 1).mean()
    false_alarm_rate = (alerted["delta"] <= -3).mean()
    miss_late_rate = df["delta"].isna().mean() + (alerted["delta"] > 1).mean()

    avg_queries = df["queries"].mean()
    avg_reward = df["reward"].mean()

    return {
        "timely_alert_rate": timely_alert_rate,
        "false_alarm_rate": false_alarm_rate,
        "miss_late_rate": miss_late_rate,
        "avg_queries": avg_queries,
        "avg_reward": avg_reward,
        "timing_error_distribution": timing_errors.describe(),
        "raw": df
    }



In [ ]:
# ============================================================
# Run evaluation on BOTH complete + partial preprocessed sets
# ============================================================

# Preprocess raw test sets
test_complete_df_prep = preprocess_test_df(test_complete_df, train_stats)
test_partial_df_prep  = preprocess_test_df(test_partial_df,  train_stats)

# Datetime formatting
test_complete_df_prep["day"] = pd.to_datetime(test_complete_df_prep["day"])
test_partial_df_prep["day"]  = pd.to_datetime(test_partial_df_prep["day"])

# Evaluate on *preprocessed* sets
tau_risk = 0.4
tau_onset = 0.5
entropy_cut = 1
results_complete = evaluate_threshold(test_complete_df_prep, pomdp_model, tau_risk, tau_onset, entropy_cut)
results_partial  = evaluate_threshold(test_partial_df_prep,  pomdp_model, tau_risk, tau_onset, entropy_cut)

print("\n===== THRESHOLD POLICY RESULTS: COMPLETE TEST SET =====")
display(results_complete)

print("\n===== THRESHOLD POLICY RESULTS: PARTIAL TEST SET =====")
display(results_partial)


In [ ]:
#grid search
tau_risk_grid   = [0.3, 0.35, 0.4, 0.45, 0.5]
tau_onset_grid  = [0.15, 0.2, 0.25, 0.3]
entropy_grid    = [1.0, 1.1, 1.2, 1.3, 1.4]

search_results = []

print("Running grid search...\n")

for tau_risk in tau_risk_grid:
    for tau_onset in tau_onset_grid:
        for entropy_cut in entropy_grid:

            metrics = evaluate_threshold(
                test_complete_df_prep,
                pomdp_model,
                tau_risk,
                tau_onset,
                entropy_cut
            )

            if metrics is None:
                continue

            search_results.append({
                "tau_risk": tau_risk,
                "tau_onset": tau_onset,
                "entropy_cut": entropy_cut,
                **metrics
            })

search_results_df = pd.DataFrame(search_results)

print(f"Total combinations evaluated: {len(search_results_df)}")
display(search_results_df.head())


# ============================================================
# Select Best Parameter Settings
# (1) Filter = false alarms < 20%
# (2) Maximize timely alert rate
# ============================================================

# Safety: ensure required columns exist
required_cols = ["false_alarm_rate", "timely_alert_rate"]
missing_cols = [c for c in required_cols if c not in search_results_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in results: {missing_cols}")

filtered = search_results_df[search_results_df["false_alarm_rate"] < 0.20]

if len(filtered) == 0:
    print("\n⚠️ No parameter sets satisfy false-alarm < 20%. Showing top performers instead.\n")
    filtered = search_results_df.sort_values("false_alarm_rate").head(10)

# Best by timely alert rate
best_timely = filtered.loc[filtered["timely_alert_rate"].idxmax()]

print("\n================ BEST PARAMETER SET ================\n")
print(best_timely)

# ============================================================
# (Optional) Sort alternatives by tradeoff
# ============================================================

sorted_candidates = filtered.sort_values(
    ["timely_alert_rate", "avg_reward"],
    ascending=[False, False]
)

print("\nTop 10 candidates by timely alert rate and reward:\n")
display(sorted_candidates.head(10))



In [ ]:
# =============================================================
# PBVI Step 1: Collect Belief Points
# =============================================================

def initial_belief_from_cycle(cycle_df):
    """Your initial belief assumption: (phase0 × risk uniform)."""
    S = len(state_list)
    first_phase = cycle_df["phase"].iloc[0]
    b = np.zeros(S)
    b[state_index[(first_phase,0)]] = 0.5
    b[state_index[(first_phase,1)]] = 0.5
    return b

belief_points = []

def collect_beliefs(test_df_prep):
    all_beliefs = []
    
    for _, cycle in test_df_prep.groupby("cycle_id"):
        b = initial_belief_from_cycle(cycle)
        all_beliefs.append(b.copy())
        
        for _, row in cycle.iterrows():
            obs = {
                "biometrics": row["biometrics"],
                "symptom": None if pd.isna(row["risk_obs"]) else int(row["risk_obs"])
            }
            
            # Predict under WAIT
            T = pomdp_model["T"]["WAIT"]
            b_pred = predict_belief(b, T)
            
            # Update belief
            b = update_belief(b_pred, obs, pomdp_model)
            all_beliefs.append(b.copy())
            
    return all_beliefs

belief_points = collect_beliefs(test_complete_df_prep)
print(f"Collected {len(belief_points)} belief points.")


In [ ]:
# =============================================================
# PBVI Step 2: Sample transitions + observations
# =============================================================

def sample_transition_and_observation(s, a, pomdp_model):
    """Sample s' and observation o = (biometrics, symptom)."""
    
    # Sample next state
    T = pomdp_model["T"][a]
    s_prime = np.random.choice(len(state_list), p=T[s])
    ph_prime, r_prime = state_list[s_prime]
    
    # Sample biometric observation ~ Gaussian
    mean = pomdp_model["emissions"]["biometric"][ph_prime]["mean"]
    cov  = pomdp_model["emissions"]["biometric"][ph_prime]["cov"]
    biom = np.random.multivariate_normal(mean, cov).flatten()
    
    # Sample symptom ~ Bernoulli
    p_sym = pomdp_model["emissions"]["symptom"][(ph_prime, r_prime)]
    symptom = 1 if np.random.rand() < p_sym else 0
    
    obs = {"biometrics": biom, "symptom": symptom}
    return s_prime, obs


In [ ]:
# =============================================================
# PBVI Step 3: Backup operator for one action
# =============================================================

def backup_alpha(b, a, alpha_vectors, pomdp_model, n_samples=25, gamma=0.98):
    S = len(state_list)
    alpha = np.zeros(S)
    
    # -----------------------------
    # Immediate expected reward
    # -----------------------------
    R_sa = np.zeros(S)
    for s in range(S):
        if a == "ALERT":
            # ALERT reward is timing-based → here approximate with 0
            R_sa[s] = 0
        else:
            R_sa[s] = pomdp_model["reward"][a]
    alpha += R_sa
    
    # -----------------------------
    # Future expected value (sampled)
    # -----------------------------
    for _ in range(n_samples):
        # Sample s from belief
        s = np.random.choice(S, p=b)
        
        # Sample transition + observation
        s_prime, obs = sample_transition_and_observation(s, a, pomdp_model)
        
        # Full Bayesian belief update from b, not from s
        T = pomdp_model["T"][a]
        b_pred = b @ T
        b_next = update_belief(b_pred, obs, pomdp_model)
        
        # Evaluate V(b_next)
        values = [b_next @ a_vec for a_vec in alpha_vectors]
        V_next = max(values)
        
        # Add discounted contribution
        alpha += (gamma * V_next) / n_samples
    
    return alpha


In [ ]:
# =============================================================
# PBVI Step 4: Backup for all actions at belief b
# =============================================================

ACTIONS = list(pomdp_model["T"].keys())

def pbvi_backup_for_belief(b, alpha_vectors, pomdp_model):
    best_alpha = None
    best_val = -np.inf
    
    for a in ACTIONS:
        alpha_a = backup_alpha(b, a, alpha_vectors, pomdp_model)
        val = b @ alpha_a
        if val > best_val:
            best_val = val
            best_alpha = alpha_a
            
    return best_alpha


In [ ]:
# =============================================================
# PBVI Step 5: One full PBVI iteration
# =============================================================

def pbvi_iteration(belief_points, alpha_vectors, pomdp_model):
    new_alphas = []
    for b in belief_points:
        new_alphas.append(pbvi_backup_for_belief(b, alpha_vectors, pomdp_model))
    return new_alphas


In [ ]:
# =============================================================
# Run PBVI
# =============================================================

alpha_vectors = [np.zeros(len(state_list))]  # V_0

K = 8  # number of PBVI iterations

for it in range(K):
    alpha_vectors = pbvi_iteration(belief_points, alpha_vectors, pomdp_model)
    print(f"PBVI iteration {it+1}/{K} complete.")


In [ ]:
# =============================================================
# PBVI Policy: Choose action maximizing α·b
# =============================================================

def pbvi_policy(b, alpha_vectors, pomdp_model):
    best_a = None
    best_val = -np.inf
    
    for a in ACTIONS:
        alpha_a = backup_alpha(b, a, alpha_vectors, pomdp_model)
        val = b @ alpha_a
        if val > best_val:
            best_val = val
            best_a = a

    return best_a


In [ ]:
def run_cycle_pbvi(cycle_df, pomdp_model, alpha_vectors):

    S = len(state_list)
    first_phase = cycle_df["phase"].iloc[0]

    b = np.zeros(S)
    b[state_index[(first_phase,0)]] = 0.5
    b[state_index[(first_phase,1)]] = 0.5

    alerted_day = None
    queries = 0
    reward = 0

    true_onset_days = cycle_df.loc[cycle_df["true_onset"] == 1, "day"]
    if len(true_onset_days) == 0:
        return None
    true_onset_day = pd.to_datetime(true_onset_days.iloc[0])

    for _, row in cycle_df.iterrows():
        a = pbvi_policy(b, alpha_vectors, pomdp_model)

        if a == "QUERY":
            queries += 1

        if a != "ALERT":
            reward += pomdp_model["reward"][a]
        else:
            delta = (row["day"] - true_onset_day).days
            reward += pomdp_model["reward"]["ALERT"](delta)
            if alerted_day is None:
                alerted_day = row["day"]

        # Predict & update belief
        T = pomdp_model["T"][a]
        b_pred = predict_belief(b, T)

        obs = {
            "biometrics": row["biometrics"],
            "symptom": None if pd.isna(row["risk_obs"]) else int(row["risk_obs"])
        }

        b = update_belief(b_pred, obs, pomdp_model)

    delta = None if alerted_day is None else (alerted_day - true_onset_day).days

    return {
        "cycle_id": cycle_df["cycle_id"].iloc[0],
        "alert_day": alerted_day,
        "delta": delta,
        "queries": queries,
        "reward": reward
    }


In [ ]:
results_complete_pbvi = []
for cid, cdf in test_complete_df_prep.groupby("cycle_id"):
    out = run_cycle_pbvi(cdf, pomdp_model, alpha_vectors)
    if out is not None:
        results_complete_pbvi.append(out)

pd.DataFrame(results_complete_pbvi)
